# 8. 비지도학습과 모델 평가

> **제8장** · **이론편 대응: 4.4절(SVD·PCA), 8.1절(학습 방식), 9.2·9.3·9.5절(앙상블·평가)**
> **예상 소요**: 50분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

지금까지는 정답(레이블)이 있는 데이터를 다뤘다. 이번에는 **정답 없이 구조를 찾는** 방법을 본다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 지도 vs 비지도 | 8.1절 |
| 2 | **K-Means 직접 구현** → sklearn 비교 | 8.3절 |
| 3 | 군집 개수 정하기 — 엘보우 | — |
| 4 | **PCA로 03번 결과 재현** | 4.4절 |
| 5 | 앙상블 — Random Forest | 9.5절 |
| 6 | 교차검증 | 9.4절 |

2절에서는 **직접 구현한 결과가 sklearn과 소수점까지 같은지** 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료")

---

## 1. 지도학습과 비지도학습 — 이론편 8.1절

이론편 8.1절에서 다룬 구분을 코드 관점에서 정리하면 이렇다.

| 구분 | 데이터 | sklearn 호출 | 예 |
|---|---|---|---|
| 지도학습 | `X`와 `y` | `fit(X, y)` | 분류, 회귀 (07장) |
| 비지도학습 | `X`만 | `fit(X)` | 군집, 차원 축소 (이 장) |

**호출 방식만 봐도 구분된다.** `fit`에 `y`를 넘기지 않으면 비지도학습이다.

비지도학습이 유용한 이유는 **레이블을 다는 데 비용이 크기 때문**이다. 이미지 100만 장에
사람이 일일이 이름을 붙이는 것보다, 비슷한 것끼리 묶어 두고 대표만 확인하는 편이 싸다.

---

## 2. K-Means 직접 구현 — 이론편 8.3절 ★

K-Means의 절차는 단순하다. 두 단계를 반복할 뿐이다.

1. **할당** — 각 점을 가장 가까운 중심에 배정한다
2. **갱신** — 각 군집의 평균으로 중심을 옮긴다

중심이 더 이상 움직이지 않으면 멈춘다. 먼저 직접 구현하고, sklearn과 비교한다.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

# 세 덩어리로 나뉜 데이터
X, y_true = make_blobs(n_samples=300, centers=3,
                       cluster_std=1.0, random_state=42)

def kmeans_manual(X, k, seed=0, max_iters=100, verbose=False):
    """K-Means 직접 구현 (이론편 8.3절)"""
    rng = np.random.RandomState(seed)

    # 초기 중심: 데이터에서 k개를 무작위로 고른다
    centers = X[rng.choice(len(X), k, replace=False)].copy()

    for it in range(max_iters):
        # --- 1단계: 할당 ---
        # 모든 점과 모든 중심 사이의 거리를 한 번에 계산 (브로드캐스팅)
        #   X[:, None, :]  → (n, 1, d)
        #   centers[None]  → (1, k, d)
        #   차이           → (n, k, d)
        distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = distances.argmin(axis=1)

        # --- 2단계: 갱신 ---
        new_centers = np.array([
            X[labels == j].mean(axis=0) if (labels == j).any() else centers[j]
            for j in range(k)
        ])

        shift = np.abs(new_centers - centers).max()
        if verbose:
            print(f"  반복 {it+1:2}: 중심 이동량 {shift:.6f}")

        if np.allclose(new_centers, centers):
            if verbose:
                print(f"  → {it+1}회 만에 수렴")
            break
        centers = new_centers

    # 관성(inertia): 각 점과 자기 중심 사이 거리 제곱의 합
    inertia = ((X - centers[labels]) ** 2).sum()
    return labels, centers, inertia


print("=" * 55)
print("K-Means 직접 구현")
print("=" * 55)
labels_m, centers_m, inertia_m = kmeans_manual(X, k=3, seed=0, verbose=True)
print()
print(f"관성(inertia) : {inertia_m:.4f}")
print(f"군집별 개수    : {[int((labels_m==j).sum()) for j in range(3)]}")

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# sklearn과 비교
# ── KMeans 파라미터 ──────────────────────────────────────────
#   n_clusters   군집 개수.  기본값 8
#                엘보우·실루엣으로 정한다 (이 장 참조)
#   init         초기화 방법.  기본값 'k-means++'
#                'random' 보다 안정적 — 초기 중심을 퍼뜨린다
#   n_init       초기화 반복 횟수.  기본값 'auto' (10 또는 1)
#                여러 번 시도해 가장 좋은 결과를 고른다
#   max_iter     최대 반복.  기본값 300
#   tol          수렴 판정 기준.  기본값 1e-4
#   random_state 난수 시드.  기본값 None → 고정 권장
# ──────────────────────────────────────────────────────────────
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)

print("=" * 55)
print("직접 구현 vs scikit-learn")
print("=" * 55)
print(f"{'':16}{'관성(inertia)':<20}{'군집 크기'}")
print("-" * 55)
print(f"{'직접 구현':16}{inertia_m:<20.4f}{sorted([int((labels_m==j).sum()) for j in range(3)])}")
print(f"{'scikit-learn':16}{km.inertia_:<20.4f}{sorted([int((km.labels_==j).sum()) for j in range(3)])}")
print("-" * 55)

diff_pct = abs(inertia_m - km.inertia_) / km.inertia_ * 100
print(f"관성 차이: {diff_pct:.4f}%")
assert diff_pct < 0.01, "결과가 크게 다릅니다"
print("[OK] 직접 구현이 sklearn과 같은 답에 도달했다")
print()
print("군집 번호는 다를 수 있다 — 어느 덩어리를 0장이라 부르는지는 임의이기 때문이다.")
print("중요한 것은 '어떻게 나눴는가'이고, 그것은 관성으로 비교한다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
colors = ["#1E40AF", "#EA580C", "#0D9488"]

# --- (1) 원본 데이터 (정답 없이 보면 이렇다) ---
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], s=20, color="#64748B", alpha=0.6)
ax.set_title("원본 데이터 — 레이블 없음")
ax.grid(alpha=0.3)

# --- (2) 직접 구현 결과 ---
ax = axes[1]
for j in range(3):
    m = labels_m == j
    ax.scatter(X[m, 0], X[m, 1], s=20, color=colors[j], alpha=0.6, label=f"군집 {j}")
ax.scatter(centers_m[:, 0], centers_m[:, 1], s=250, marker="*",
           color="black", edgecolor="white", linewidth=1.5, zorder=5, label="중심")
ax.set_title(f"직접 구현 (관성 {inertia_m:.1f})")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (3) sklearn 결과 ---
ax = axes[2]
for j in range(3):
    m = km.labels_ == j
    ax.scatter(X[m, 0], X[m, 1], s=20, color=colors[j], alpha=0.6)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], s=250,
           marker="*", color="black", edgecolor="white", linewidth=1.5, zorder=5)
ax.set_title(f"scikit-learn (관성 {km.inertia_:.1f})")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("색만 다를 뿐 같은 방식으로 나뉘었다.")

### 초기 중심에 따라 결과가 달라진다

K-Means는 **초기 중심을 어디에 두느냐에 따라 다른 답에 도달할 수 있다.**
이론편 5.5절에서 다룬 국소 최솟값 문제와 같은 구조다.

sklearn의 `n_init=10`은 "서로 다른 초기값으로 10번 돌려서 가장 좋은 것을 고른다"는 뜻이다.
직접 구현에서도 시드를 바꿔 가며 확인해 보자.

In [ ]:
import numpy as np

print("=" * 55)
print("초기값(시드)에 따른 결과 차이")
print("=" * 55)
print(f"{'시드':<8}{'관성':<16}{'군집 크기'}")
print("-" * 55)

results = []
for seed in range(6):
    lab, cen, inr = kmeans_manual(X, k=3, seed=seed)
    sizes = sorted([int((lab == j).sum()) for j in range(3)])
    results.append(inr)
    mark = ""
    print(f"{seed:<8}{inr:<16.4f}{sizes}{mark}")

print("-" * 55)
print(f"최선: {min(results):.4f}  /  최악: {max(results):.4f}")
if max(results) - min(results) > 1:
    print("→ 초기값에 따라 결과가 달라졌다. 여러 번 돌려 최선을 고르는 이유다.")
else:
    print("→ 이 데이터는 덩어리가 뚜렷해 초기값의 영향이 작았다.")

---

## 3. 군집을 몇 개로 나눌 것인가 — 엘보우

K-Means는 **k를 사람이 정해 줘야 한다.** 데이터가 몇 개 덩어리인지 모를 때 어떻게 할까.

가장 흔한 방법이 **엘보우(elbow)**다. k를 늘려 가며 관성을 재고, 그래프가 꺾이는 지점을 고른다.

관성은 k가 커질수록 무조건 줄어든다(극단적으로 모든 점이 자기 중심이면 0이다).
따라서 "많이 줄어드는 구간이 끝나는 지점"을 찾는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

ks = range(1, 9)
inertias = []
for k in ks:
    inertias.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_)

print("=" * 50)
print("k별 관성")
print("=" * 50)
print(f"{'k':<6}{'관성':<16}{'전 단계 대비 감소'}")
print("-" * 50)
for i, (k, inr) in enumerate(zip(ks, inertias)):
    if i == 0:
        print(f"{k:<6}{inr:<16.1f}—")
    else:
        drop = inertias[i-1] - inr
        pct = drop / inertias[i-1] * 100
        mark = "  ← 여기까지 크게 준다" if pct > 50 else ""
        print(f"{k:<6}{inr:<16.1f}{drop:8.1f} ({pct:.1f}%){mark}")
print("-" * 50)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(ks), inertias, marker="o", linewidth=2, color="#1E40AF")
ax.axvline(3, color="#EA580C", linestyle="--", linewidth=1.5)
ax.annotate("여기서 꺾인다 (k=3)", xy=(3, inertias[2]),
            xytext=(4.5, inertias[1]), fontsize=10, color="#EA580C",
            arrowprops=dict(arrowstyle="->", color="#EA580C"))
ax.set_xlabel("군집 개수 k")
ax.set_ylabel("관성 (inertia)")
ax.set_title("엘보우 방법")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("k=3에서 꺾인다. 실제로 데이터를 3개 덩어리로 만들었으므로 맞는 판단이다.")
print("다만 실제 데이터에서는 이렇게 뚜렷하지 않은 경우가 많다.")

---

## 4. PCA — 이론편 4.4절 값 검증 ★

03장에서 공분산 행렬의 고유분해로 PCA를 직접 계산했다.
sklearn으로 하면 같은 결과가 나오는지 확인한다.

03장에서 얻은 값: **설명 분산비 [97.67%, 2.33%]**

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# 03장과 완전히 같은 데이터
rng = np.random.RandomState(42)
n = 200
x1 = rng.randn(n) * 2.0
x2 = x1 * 0.8 + rng.randn(n) * 0.5
X_pca = np.stack([x1, x2], axis=1)

pca = PCA(n_components=2).fit(X_pca)
ratio = pca.explained_variance_ratio_ * 100

print("=" * 55)
print("PCA — 03장 결과와 대조")
print("=" * 55)
print(f"sklearn 설명 분산비 : {ratio.round(2)} %")
print(f"03번 고유분해 결과  : [97.67  2.33] %")
print()
print(f"제1주성분 방향      : {pca.components_[0].round(4)}")
print()

assert abs(ratio[0] - 97.67) < 0.05, "03번 결과와 다릅니다"
print("[OK] 03장(고유분해)과 일치")
print()
print("sklearn의 PCA는 내부적으로 SVD를 쓴다.")
print("이론편 4.4절에서 '고유분해와 SVD는 같은 것을 계산한다'고 한 그대로다.")

### 이론편 4.4절 성적표로도 확인

이론편 4.4절의 학생 성적표 예제를 sklearn PCA로 압축해 본다.
손으로 구했던 값은 **보존율 99.69%, 최대 오차 0.47점**이었다.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# 이론편 4.4절 성적표
B = np.array([[90.0, 88.0, 92.0],
              [72.0, 70.0, 74.0],
              [84.0, 82.0, 86.0],
              [60.0, 58.0, 62.0]])

pca1 = PCA(n_components=1).fit(B)
compressed = pca1.transform(B)            # 3차원 → 1차원
restored = pca1.inverse_transform(compressed)   # 다시 3차원으로

print("=" * 55)
print("이론편 4.4절 성적표 — PCA로 압축")
print("=" * 55)
print("원본")
print(B)
print()
print(f"압축 결과 (학생당 숫자 1개)")
print(compressed.round(2).ravel())
print()
print("복원")
print(restored.round(1))
print()

max_err = np.abs(B - restored).max()
print(f"설명 분산비 : {pca1.explained_variance_ratio_[0]*100:.4f} %")
print(f"최대 오차   : {max_err:.4f} 점")
print(f"이론편 SVD 결과: 보존율 99.69%, 최대 오차 0.47점")
print()
print("PCA는 평균을 뺀 뒤 계산하므로 이론편의 순수 SVD와 수치가 조금 다르다.")
print("하지만 '소수의 방향이 정보 대부분을 담는다'는 결론은 같다.")

---

## 5. 앙상블 — Random Forest (이론편 9.5절)

09장에서 결정 트리 하나를 썼을 때 과대적합이 심했다. 이론편 9.5절에서 다룬 앙상블은
**여러 모델의 결과를 합쳐** 이 문제를 완화한다.

Random Forest는 다음 두 가지로 트리들을 서로 다르게 만든다.

1. 각 트리를 **데이터의 일부**로 학습한다
2. 각 분기에서 **특성의 일부**만 후보로 본다

그러면 트리마다 다른 실수를 하게 되고, 여러 개를 평균 내면 실수가 상쇄된다.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.3, random_state=42, stratify=ym)

models = {
    "결정 트리 1개": DecisionTreeClassifier(random_state=42),
    # ── RandomForestClassifier 파라미터 ──────────────────────────
    #   n_estimators   트리 개수.  기본값 100
    #                  많을수록 안정적(과대적합 위험 없음). 예: 100~500
    #   max_depth      각 트리의 최대 깊이.  기본값 None
    #                  랜덤 포레스트는 보통 제한하지 않는다
    #   max_features   분할 시 볼 특성 수.  기본값 'sqrt' (분류)
    #                  회귀는 1.0(전부). 작을수록 트리가 서로 달라진다
    #   bootstrap      복원 추출 사용.  기본값 True
    #   oob_score      OOB 평가 사용.  기본값 False
    #                  True 로 하면 별도 검증셋 없이 성능 추정 가능
    #   n_jobs         병렬 처리 코어 수.  기본값 None(1개)
    #                  -1 로 하면 모든 코어 사용
    #   class_weight   불균형 대응.  기본값 None → 'balanced' 고려
    # ──────────────────────────────────────────────────────────────
    "Random Forest (10그루)": RandomForestClassifier(n_estimators=10, random_state=42),
    "Random Forest (100그루)": RandomForestClassifier(n_estimators=100, random_state=42),
}

print("=" * 68)
print("단일 트리 vs 앙상블 (이론편 9.5절)")
print("=" * 68)
print(f"{'모델':<26}{'훈련':<12}{'시험':<12}{'격차'}")
print("-" * 68)
for name, m in models.items():
    m.fit(Xm_tr, ym_tr)
    tr = m.score(Xm_tr, ym_tr)
    te = m.score(Xm_te, ym_te)
    gap = tr - te
    note = "  ← 과대적합" if gap > 0.12 else ""
    print(f"{name:<26}{tr:<12.4f}{te:<12.4f}{gap:.4f}{note}")
print("-" * 68)
print()
print("트리 하나는 훈련 정확도가 1.0인데 시험은 낮다 — 외운 것이다.")
print("여러 그루를 합치면 훈련 정확도는 비슷하지만 시험 성능이 오른다.")

### 특성 중요도

Random Forest는 **어느 특성이 판단에 많이 쓰였는지** 알려준다.
이론편 9.1절에서 다룬 해석 가능성이 부분적으로 유지되는 셈이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 특성이 여러 개인 데이터
wine = load_wine()
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(
    wine.data, wine.target, test_size=0.3, random_state=42, stratify=wine.target)

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xw_tr, yw_tr)

print("=" * 55)
print("특성 중요도 (와인 분류)")
print("=" * 55)
print(f"특성 개수 : {len(wine.feature_names)}")
print(f"시험 정확도: {rf.score(Xw_te, yw_te):.4f}")
print()

order = np.argsort(rf.feature_importances_)[::-1]
print("중요도 상위 5개")
for i in order[:5]:
    print(f"  {wine.feature_names[i]:<32}{rf.feature_importances_[i]:.4f}")
print()
print("하위 3개")
for i in order[-3:]:
    print(f"  {wine.feature_names[i]:<32}{rf.feature_importances_[i]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
top = order[:8][::-1]
ax.barh([wine.feature_names[i] for i in top],
        [rf.feature_importances_[i] for i in top], color="#0D9488")
ax.set_xlabel("중요도")
ax.set_title("특성 중요도 상위 8개")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. 교차검증 — 이론편 9.4절

07장에서 데이터를 훈련·검증·시험으로 나눴다. 그런데 데이터가 적으면 문제가 생긴다.
**어떻게 나누느냐에 따라 성능이 크게 달라지기 때문**이다.

교차검증은 이를 완화한다. 데이터를 k조각으로 나누고, 각 조각을 한 번씩 검증용으로 쓰며
k번 학습·평가한 뒤 평균을 낸다.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold

digits = load_digits()
print("=" * 55)
print("교차검증 (이론편 9.4절)")
print("=" * 55)
print(f"데이터: 손글씨 숫자 {digits.data.shape[0]}개, 특성 {digits.data.shape[1]}개")
print()

model = RandomForestClassifier(n_estimators=50, random_state=42)
scores = cross_val_score(model, digits.data, digits.target, cv=5)

print("5겹 교차검증 결과")
print("-" * 55)
for i, s in enumerate(scores, 1):
    bar = "█" * int(s * 40)
    print(f"  {i}번째 : {s:.4f}  {bar}")
print("-" * 55)
print(f"  평균   : {scores.mean():.4f}")
print(f"  표준편차: {scores.std():.4f}")
print()
print(f"이 모델의 성능은 대략 {scores.mean():.3f} ± {scores.std():.3f} 라고 말할 수 있다.")
print()
print("한 번만 나눠 평가했다면 운에 따라")
print(f"  {scores.min():.4f} 이 나올 수도, {scores.max():.4f} 가 나올 수도 있었다.")
print("교차검증은 이 편차를 함께 알려준다.")

### 교차검증을 쓸 때 주의할 점

**표준화는 각 겹 안에서 해야 한다.** 전체 데이터로 먼저 표준화하고 교차검증을 돌리면,
검증 조각의 정보가 표준화 통계에 섞여 들어간다(데이터 누수).

sklearn에서는 `Pipeline`으로 이를 자동 처리할 수 있다.

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_digits

digits = load_digits()

# 잘못된 방법: 전체 데이터로 먼저 표준화 (데이터 누수)
X_leaked = StandardScaler().fit_transform(digits.data)
score_leaked = cross_val_score(LogisticRegression(max_iter=2000),
                               X_leaked, digits.target, cv=5)

# 올바른 방법: Pipeline — 각 겹 안에서 표준화
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000)),
])
score_correct = cross_val_score(pipe, digits.data, digits.target, cv=5)

print("=" * 55)
print("데이터 누수 비교")
print("=" * 55)
print(f"{'방식':<28}{'평균 점수':<14}{'표준편차'}")
print("-" * 55)
print(f"{'전체 표준화 후 CV (누수)':<28}{score_leaked.mean():<14.4f}{score_leaked.std():.4f}")
print(f"{'Pipeline 사용 (올바름)':<28}{score_correct.mean():<14.4f}{score_correct.std():.4f}")
print("-" * 55)
print()
print("이 데이터에서는 차이가 작지만, 특성이 적거나 데이터가 작으면 차이가 커진다.")
print("습관적으로 Pipeline을 쓰는 것이 안전하다.")

---

## 7. 정리

### 확인한 값

| 대조 대상 | 내용 | 결과 |
|---|---|---|
| 직접 구현 ↔ sklearn | K-Means 관성 | 소수점까지 일치 ✓ |
| 03장 | PCA 설명 분산비 97.67% | 일치 ✓ |
| 이론편 4.4절 | 성적표 압축 | 결론 일치 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 비지도학습 | `fit(X)` — `y`가 없다 |
| K-Means | 할당 ↔ 갱신 반복, 초기값에 따라 결과가 달라짐 |
| `n_init` | 여러 초기값 중 최선을 고름 |
| 엘보우 | 관성이 꺾이는 k를 고름 |
| PCA | sklearn은 SVD 사용 — 03번 고유분해와 같은 결과 |
| 앙상블 | 여러 모델을 합쳐 과대적합 완화 |
| 교차검증 | 성능의 **평균과 편차**를 함께 봄 |
| **Pipeline** | 데이터 누수 방지 — 습관적으로 사용 |

### 여기까지 온 지점

07~08장에서 라이브러리로 머신러닝을 다뤄 봤다. 하지만 이 모델들은 모두
**사람이 특성을 정해 준** 경우에만 작동한다. 이미지의 픽셀을 그대로 넣으면 잘 되지 않는다.

이론편 12장에서 다룬 신경망은 **특성 자체를 학습**한다. 다음 장부터 그것을 직접 만든다.

### 다음 장

**9. 결정트리와 앙상블** — 이론편 9.1~9.5절.
스무고개처럼 질문을 이어가며 데이터를 나누는 결정 트리의 원리를 손으로 계산해 보고, 여러 트리를 모아 성능을 높이는 배깅과 부스팅을 직접 구현한다.